In [1]:
"""Complete training pipeline for defect detection models."""

import os
import sys
from pathlib import Path

# Fix the import paths
from abbvisionsystem.training_pipeline.data_manager import organize_dataset, prepare_yolo_dataset, generate_synthetic_defects
from abbvisionsystem.training_pipeline.yolo_trainer import YOLODefectDetector, create_multi_object_test_images
from abbvisionsystem.training_pipeline.resnet_trainer import DefectClassificationModel

def run_complete_pipeline(
    source_data_dir: str,
    use_yolo: bool = True,
    use_classification: bool = True,
    train_yolo_epochs: int = 100,
    train_classification_epochs: int = 50
):
    """Run complete training pipeline for both YOLO and classification models."""
    
    print("🚀 Starting Complete Defect Detection Training Pipeline")
    print("=" * 60)
    
    # Step 1: Organize dataset
    print("\n📁 Step 1: Organizing dataset...")
    classification_dataset = "defect_detection_dataset"
    organize_dataset(source_data_dir, classification_dataset)
    
    # Step 1.5: Create realistic training data with backgrounds
    print("\n🎨 Step 1.5: Creating realistic training data...")
    from abbvisionsystem.training_pipeline.data_manager import augment_with_backgrounds, prepare_yolo_dataset_from_realistic
    
    realistic_train_dir = "realistic_training_data"
    augment_with_backgrounds(
        source_data_dir,
        realistic_train_dir,
        objects_per_image=(1, 3),
        images_per_object=3,
        multi_object_scenes=100
    )
    
    # Step 2: Prepare YOLO dataset with realistic data
    print("\n🎯 Step 2: Preparing YOLO dataset from realistic data...")
    yolo_dataset_yaml = prepare_yolo_dataset_from_realistic(realistic_train_dir, "yolo_dataset_realistic")
    
    # Step 3: Create multi-object test images
    print("\n🖼️ Step 3: Creating multi-object test images...")
    create_multi_object_test_images(
        f"{classification_dataset}/test",
        "multi_object_test",
        images_per_composition=30
    )
    
    results = {}
    
    # Step 4: Train YOLO model (recommended for your use case)
    if use_yolo:
        print("\n🤖 Step 4: Training YOLOv8 model...")
        yolo_detector = YOLODefectDetector()
               
        try:
            best_yolo_weights = yolo_detector.train(
                dataset_yaml=yolo_dataset_yaml,
                epochs=train_yolo_epochs,
                imgsz=640,
                batch=16,
                project='trained_models',
                name='yolo_defect_detector'
                # Removed deprecated parameters: patience, save_period
            )
            
            # Evaluate on your "both" dataset
            print("\n📊 Evaluating YOLO model on real multi-object images...")
            yolo_results = evaluate_on_both_dataset(yolo_detector, f"{source_data_dir}/both")
            results['yolo'] = yolo_results
            
        except Exception as e:
            print(f"YOLO training failed: {e}")
            results['yolo'] = None
    
    # Step 5: Train classification model (for comparison)
    if use_classification:
        print("\n🧠 Step 5: Training ResNet50V2 classification model...")
        classifier = DefectClassificationModel()
        classifier.build_model()
        
        try:
            # Prepare data
            train_gen, val_gen = classifier.prepare_data_generators(
                f"{classification_dataset}/train",
                f"{classification_dataset}/validation"
            )
            
            # Train
            classifier.train(
                train_gen, val_gen,
                epochs=train_classification_epochs,
                model_name="resnet_defect_classifier"
            )
            
            # Evaluate - fix the test generator creation
            test_datagen = classifier.prepare_data_generators(
                f"{classification_dataset}/test",
                f"{classification_dataset}/test"  # Using same directory
            )[1]  # Use validation generator (no augmentation)
            
            classification_results = classifier.evaluate(test_datagen)
            results['classification'] = classification_results
            
            # Save model
            classifier.save_model("resnet_defect_classifier")
            
            print(f"Classification Results:")
            print(f"  Accuracy: {classification_results['test_accuracy']:.4f}")
            print(f"  Precision: {classification_results['test_precision']:.4f}")
            print(f"  Recall: {classification_results['test_recall']:.4f}")
            
        except Exception as e:
            print(f"Classification training failed: {e}")
            results['classification'] = None
    
    # Step 6: Compare models
    print("\n📈 Step 6: Model Comparison Summary")
    print("=" * 40)
    
    if results.get('yolo') and results.get('classification'):
        print("Model Performance Comparison:")
        print(f"{'Metric':<15} {'YOLO':<10} {'ResNet50V2':<12}")
        print("-" * 37)
        print(f"{'Accuracy':<15} {results['yolo']['accuracy']:<10.4f} {results['classification']['test_accuracy']:<12.4f}")
        print(f"{'Precision':<15} {results['yolo']['precision']:<10.4f} {results['classification']['test_precision']:<12.4f}")
        print(f"{'Recall':<15} {results['yolo']['recall']:<10.4f} {results['classification']['test_recall']:<12.4f}")
        
        # Calculate F1 for classification
        precision = results['classification']['test_precision']
        recall = results['classification']['test_recall']
        f1_classification = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        
        print(f"{'F1 Score':<15} {results['yolo']['f1_score']:<10.4f} {f1_classification:<12.4f}")
    
    print("\n✅ Pipeline completed successfully!")
    print("\n🎯 RECOMMENDATION FOR YOUR USE CASE:")
    print("Since you need to detect multiple objects in real-world images,")
    print("YOLOv8 is the better choice as it can:")
    print("  • Detect multiple objects simultaneously")
    print("  • Provide bounding box locations")
    print("  • Handle varying numbers of objects per image")
    print("  • Scale better to production environments")
    
    return results


# Test function to check if everything is properly set up
def test_pipeline_setup():
    """Test if all components are properly set up."""
    print("🔍 Testing pipeline setup...")
    
    try:
        from abbvisionsystem.training_pipeline.data_manager import organize_dataset
        print("✅ data_manager import successful")
    except ImportError as e:
        print(f"❌ data_manager import failed: {e}")
        return False
    
    try:
        from abbvisionsystem.training_pipeline.yolo_trainer import YOLODefectDetector
        print("✅ yolo_trainer import successful")
    except ImportError as e:
        print(f"❌ yolo_trainer import failed: {e}")
        return False
    
    try:
        from abbvisionsystem.training_pipeline.resnet_trainer import DefectClassificationModel
        print("✅ resnet_trainer import successful")
    except ImportError as e:
        print(f"❌ resnet_trainer import failed: {e}")
        return False
    
    # Check if ultralytics is available for YOLO
    try:
        from ultralytics import YOLO
        print("✅ ultralytics available")
    except ImportError:
        print("⚠️  ultralytics not installed. Install with: pip install ultralytics")
    
    # Check if tensorflow is available
    try:
        import tensorflow as tf
        print(f"✅ tensorflow {tf.__version__} available")
    except ImportError:
        print("❌ tensorflow not installed")
        return False
    
    print("✅ Pipeline setup test completed successfully!")
    return True


if __name__ == "__main__":
    # First test the setup
    if not test_pipeline_setup():
        print("❌ Setup test failed. Please fix the issues above.")
        exit(1)
    
    # Run the complete pipeline
    source_dir = "data/choco-pie"  # Update this path
    
    if not os.path.exists(source_dir):
        print(f"Source directory {source_dir} not found!")
        print("Please update the source_dir variable to point to your data.")
        print("Expected structure:")
        print("data/choco-pie/")
        print("├── good/")
        print("│   ├── image1.JPG")
        print("│   └── image2.JPG")
        print("└── defect/")
        print("    ├── defect1.JPG")
        print("    └── defect2.JPG")
    else:
        # Check data structure
        good_dir = os.path.join(source_dir, "good")
        defect_dir = os.path.join(source_dir, "defect")
        
        if not os.path.exists(good_dir):
            print(f"❌ 'good' directory not found in {source_dir}")
            exit(1)
        if not os.path.exists(defect_dir):
            print(f"❌ 'defect' directory not found in {source_dir}")
            exit(1)
            
        good_files = [f for f in os.listdir(good_dir) if f.endswith(('.JPG', '.jpg', '.png', '.bmp'))]
        defect_files = [f for f in os.listdir(defect_dir) if f.endswith(('.JPG', '.jpg', '.png', '.bmp'))]
        
        print(f"📊 Dataset Summary:")
        print(f"  Normal samples: {len(good_files)}")
        print(f"  Defect samples: {len(defect_files)}")
        
        if len(good_files) == 0 or len(defect_files) == 0:
            print("❌ Insufficient data. Need at least 1 image in each category.")
            exit(1)
        
        # Run pipeline
        results = run_complete_pipeline(
            source_data_dir=source_dir,
            use_yolo=True,
            use_classification=True,
            train_yolo_epochs=50,  # Reduced for testing
            train_classification_epochs=25  # Reduced for testing
        )
        
def evaluate_on_both_dataset(yolo_detector, both_images_dir):
    """Evaluate on your 'both' dataset with multiple objects."""
    results = {
        "total_images": 0,
        "images_with_detections": 0,
        "total_detections": 0,
        "avg_detections_per_image": 0.0,
        "confidence_scores": [],
    }
    
    image_files = [f for f in os.listdir(both_images_dir) 
                   if f.endswith(('.jpg', '.jpeg', '.png', '.JPG'))]
    
    for img_file in image_files:
        img_path = os.path.join(both_images_dir, img_file)
        detections = yolo_detector.predict(img_path, conf_threshold=0.25)
        
        results["total_images"] += 1
        num_detections = len(detections["boxes"])
        
        if num_detections > 0:
            results["images_with_detections"] += 1
            results["total_detections"] += num_detections
            results["confidence_scores"].extend(detections["scores"])
    
    if results["total_images"] > 0:
        results["avg_detections_per_image"] = results["total_detections"] / results["total_images"]
        results["detection_rate"] = results["images_with_detections"] / results["total_images"]
    
    return results

🔍 Testing pipeline setup...
✅ data_manager import successful
✅ yolo_trainer import successful
✅ resnet_trainer import successful
✅ ultralytics available
✅ tensorflow 2.19.0 available
✅ Pipeline setup test completed successfully!
📊 Dataset Summary:
  Normal samples: 23
  Defect samples: 26
🚀 Starting Complete Defect Detection Training Pipeline

📁 Step 1: Organizing dataset...
Dataset organized into defect_detection_dataset

🎨 Step 1.5: Creating realistic training data...
🎨 Creating realistic training data with backgrounds...
  Creating single-object training images...
  Creating multi-object training scenes...
✅ Generated 247 realistic training images with backgrounds

🎯 Step 2: Preparing YOLO dataset from realistic data...
✅ Realistic YOLO dataset prepared in yolo_dataset_realistic
   Train: 172 images
   Val: 37 images
   Test: 38 images

🖼️ Step 3: Creating multi-object test images...
📝 Creating 30 multi-object test images...
   Normal images available: 4
   Defect images available: 

train: Scanning /Users/ducle/Library/CloudStorage/OneDrive-RMITUniversity/Courses/OENG1183 - Capstone A/abb-capstone/yolo_dataset_realistic/labels/train... 172 images, 0 backgrounds, 0 corrupt: 100%|██████████| 172/172 [00:00<00:00, 3131.14it/s]

train: New cache created: /Users/ducle/Library/CloudStorage/OneDrive-RMITUniversity/Courses/OENG1183 - Capstone A/abb-capstone/yolo_dataset_realistic/labels/train.cache


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 799.7±153.7 MB/s, size: 183.7 KB)


val: Scanning /Users/ducle/Library/CloudStorage/OneDrive-RMITUniversity/Courses/OENG1183 - Capstone A/abb-capstone/yolo_dataset_realistic/labels/val... 37 images, 0 backgrounds, 0 corrupt: 100%|██████████| 37/37 [00:00<00:00, 3732.84it/s]

...

INFO:tensorflow:Assets written to: /var/folders/8w/rnt4yvg55s123wgbg_0r__cw0000gn/T/tmpwuaq1_ej/assets


INFO:tensorflow:Assets written to: /var/folders/8w/rnt4yvg55s123wgbg_0r__cw0000gn/T/tmpwuaq1_ej/assets


Saved artifact at '/var/folders/8w/rnt4yvg55s123wgbg_0r__cw0000gn/T/tmpwuaq1_ej'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_190')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  13037453392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13004601296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13003664336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13003665296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13004602640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13004598992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13003664528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  12998489680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  12998489104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  12998491600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134265

W0000 00:00:1754919936.870462 1740785 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1754919936.870725 1740785 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
I0000 00:00:1754919936.965181 1740785 mlir_graph_optimization_pass.cc:425] MLIR V1 optimization pass is not enabled


Model saved in multiple formats:
- H5: trained_models/resnet_defect_classifier.h5
- Keras: trained_models/resnet_defect_classifier.keras
- TFLite: trained_models/resnet_defect_classifier.tflite
Classification Results:
  Accuracy: 0.8889
  Precision: 0.8000
  Recall: 1.0000

📈 Step 6: Model Comparison Summary

✅ Pipeline completed successfully!

🎯 RECOMMENDATION FOR YOUR USE CASE:
Since you need to detect multiple objects in real-world images,
YOLOv8 is the better choice as it can:
  • Detect multiple objects simultaneously
  • Provide bounding box locations
  • Handle varying numbers of objects per image
  • Scale better to production environments
